# 🖥️ Manipulating Field data

Because Parcels `Field` objects are built on top of `xarray` DataArrays, you can leverage the powerful data manipulation capabilities of `xarray` to preprocess or modify your field data before using it in particle simulations. This tutorial provides some common examples of how to leverage that power - however we highly recommend also reading the Xarray documentation!

Let's start with some imports.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import parcels
import parcels.tutorial

## Looping field data and overriding the time dimension

Sometimes we want to loop field data (e.g., use one year of data again ten times), or we want to completely override the time dimension.

In previous versions of Parcels (pre version 4) this was done via Parcels function parameters `timestamps` and `time_periodic`. In this version of Parcels, we simply just manipulate the Xarray objects directly before passing the objects to Parcels.

In [ ]:
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)
ds_fields

Here we're dealing with daily data for from 2024-01-01 to 2024-02-01 (32 observations in total).

Let's loop this data for a total of 90 days.

In [ ]:
N_TIME = ds_fields.time.size
N_TIME_TARGET = 90

time_indices = np.arange(N_TIME_TARGET) % N_TIME
new_time_coordinate = np.arange(
    np.datetime64("2000-01-01"),
    np.datetime64("2000-01-01") + N_TIME_TARGET * np.timedelta64(1, "D"),
    np.timedelta64(1, "D"),
).astype("datetime64[ns]")
print(time_indices)

In [ ]:
ds_periodic = ds_fields.isel(time=time_indices)
ds_periodic["time"] = new_time_coordinate

# Now we have our periodic Dataset, let's take a look at the U velocity at a given position
# to verify
ds_periodic["uo"].isel(depth=0, latitude=0, longitude=0).plot()

From this example, its easy to see how we can completely override the time dimension. For example, by adding a linear offset to the coordinates to make the data for February (and the first few days of March).

In [ ]:
new_time_coordinate = ds_fields.time.values + np.timedelta64(31, "D")

ds_offset = ds_fields.copy()
ds_offset["time"] = new_time_coordinate
ds_offset

## Summing Fields
In some applications, you may want to sum multiple fields together to create a combined effect on particle movement. For example, you might want to combine ocean currents with wind-driven surface drift. This tutorial demonstrates how to sum multiple fields in Parcels and visualize the resulting particle trajectories.

We base this tutorial on the example provided in the [Parcels Kernel loop explanation](explanation_kernelloop.md). There, we combined two kernels to simulate the joint effect of currents and winds.

However, we can also do that in one kernel, by combining the fields directly, using `xarray` operations.

We start with loading the necessary libraries and data

In [ ]:
# Load the CopernicusMarine data in the Agulhas region from the example_datasets
ds_fields = parcels.tutorial.open_dataset(
    "CopernicusMarine_data_for_Argo_tutorial/data"
)

# Create an idealised wind field and add it to the dataset
ydim, xdim = len(ds_fields.latitude), len(ds_fields.longitude)
ds_fields["UWind"] = xr.DataArray(
    data=0.5
    * np.ones((ydim, xdim))
    * np.sin(ds_fields.latitude.values - ds_fields.latitude.values.mean())[:, None],
    coords=[ds_fields.latitude, ds_fields.longitude],
)

ds_fields["VWind"] = xr.DataArray(
    data=np.zeros((ydim, xdim)),
    coords=[ds_fields.latitude, ds_fields.longitude],
)

Now here comes the trick: we can simply sum the fields together using `xarray` operations

In [ ]:
# Combine ocean currents and wind fields
ds_fields["U"] = ds_fields["uo"] + ds_fields["UWind"]
ds_fields["V"] = ds_fields["vo"] + ds_fields["VWind"]

```{note}
Combining fields in this way assumes that the fields are defined on the same grid and have compatible dimensions. Ensure that the fields you are summing are aligned correctly to avoid unexpected results.
```

We can then run the same simulation as in the [Kernel loop explanation tutorial](explanation_kernelloop.md), but now using the combined fields. We only need the default advection kernel now, since the effects of both currents and winds are already included in the summed fields.

In [ ]:
fields = {
    "U": ds_fields["U"],
    "V": ds_fields["V"],
}
ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=fields)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

# Convert the FieldSet to windowed arrays for better performance
fieldset = fieldset.to_windowed_arrays()

npart = 10
lons = np.repeat(32.2, npart)
lats = np.linspace(-32.5, -30.5, npart)

pset = parcels.ParticleSet(fieldset, pclass=parcels.Particle, y=lats, x=lons)
output_file = parcels.ParticleFile(
    path="summed_advection_wind.parquet",
    outputdt=np.timedelta64(6, "h"),
)
pset.execute(
    [parcels.kernels.AdvectionRK2],
    runtime=np.timedelta64(5, "D"),
    dt=np.timedelta64(1, "h"),
    output_file=output_file,
)

We can then plot the trajectories, and confirm that they particles move the same as in the [Kernel loop explanation tutorial](explanation_kernelloop.md), where we combined the effects of currents and winds using two separate kernels.

In [ ]:
# Plot the resulting particle trajectories overlapped for both cases
summed_advection_wind = parcels.read_particlefile("summed_advection_wind.parquet")
fig, ax = plt.subplots(figsize=(5, 3))
for traj in summed_advection_wind.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"], traj["y"], "-")
plt.show()